# 01 · Generate OkTex Pipeline Data → Unity Catalog Delta

**Target:** `stable_classic_wg38i9_catalog.oneok_okt`  
**Warehouse:** Serverless Starter Warehouse JPao

> **Data policy:** Meter names, types, and locations are the **real** OkTex stations from the *public* OKT System Overview Map (georeferenced Esri PDF). All measurement **quantities** are **synthetic**. No ONEOK or customer operational data is used.**

This notebook generates three tables — `dim_meters`, `pipeline_segments`, `fact_daily_measurements` — and loads them into Delta, then verifies.

### 1. Generate the synthetic dataset (pure Python, seeded)

In [ ]:
from generate_data import build_meters, build_segments, build_measurements

meters = build_meters()
segments = build_segments()
measurements = build_measurements(meters)
print(f'meters={len(meters)}  route_vertices={len(segments)}  daily_measurements={len(measurements)}')
print('meter types:', sorted({m["meter_type"] for m in meters}))

meters=43  route_vertices=17  daily_measurements=2580
meter types: ['BIDIRECTIONAL', 'DELIVERY', 'RECEIPT']


### 2. Inspect a few generated rows

In [ ]:
import json
print('--- dim_meters[0] ---')
print(json.dumps(meters[0], indent=2))
print('--- fact_daily_measurements[0] ---')
print(json.dumps(measurements[0], indent=2))

--- dim_meters[0] ---
{
  "meter_id": "OKT-DN1-01",
  "meter_name": "Del Norte 1",
  "meter_type": "RECEIPT",
  "latitude": 31.752,
  "longitude": -106.443,
  "county": "El Paso",
  "state": "TX",
  "segment": "West Texas \u2014 El Paso",
  "station_group": "DEL NORTE #1",
  "pipe_diameter_in": 24,
  "capacity_dth": 145000,
  "operator": "OkTex Pipeline Company, L.L.C. (TSP 80-002-2246)",
  "status": "ACTIVE"
}
--- fact_daily_measurements[0] ---
{
  "gas_day": "2026-06-30",
  "meter_id": "OKT-DN1-01",
  "scheduled_dth": 85716,
  "actual_dth": 85485,
  "pressure_psig": 829.4,
  "temperature_f": 82.4,
  "variance_pct": -0.27
}


### 3. Create & load the Delta tables (idempotent `CREATE OR REPLACE` + insert)
Executed against the Databricks SQL warehouse via the Statement Execution API.

In [ ]:
import load_delta
load_delta.main()

# OkTex data build -> stable_classic_wg38i9_catalog.oneok_okt

## Step 1: dim_meters
inserted 43 meter rows
meter_type    | meters | total_capacity_dth
--------------+--------+-------------------
BIDIRECTIONAL | 8      | 850000            
DELIVERY      | 27     | 1475000           
RECEIPT       | 8      | 946000            

## Step 2: pipeline_segments (route polyline)
inserted 17 route vertices
seq | latitude | longitude | segment_name                
----+----------+-----------+-----------------------------
1   | 31.752   | -106.443  | West Texas — El Paso lateral
2   | 31.812   | -106.52   | West Texas — El Paso lateral
3   | 31.91    | -106.606  | West Texas — El Paso lateral
4   | 32.003   | -106.603  | West Texas — El Paso lateral
5   | 35.8     | -100.24   | Oklahoma mainline (OK-09)   

## Step 3: fact_daily_measurements
inserted 2580 daily measurement rows
## Step 4: verification
### row counts
table                   | rows
------------------------+-----
dim_meters        